In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [ ]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [2]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [3]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
    system_prompt=prompt
)

In [5]:
from langchain.messages import HumanMessage

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]}
)

In [6]:
from pprint import pprint

pprint(response)

{'messages': [HumanMessage(content='Tell me about the langchain-mcp-adapters library', additional_kwargs={}, response_metadata={}, id='d6041875-e122-4689-9991-1eccab09a6af'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "what is langchain-mcp-adapters library"}'}, '__gemini_function_call_thought_signatures__': {'call_18189680853129680105': 'EnIKcAFpFH0TcanQdHke48k1+eZRkcC5O34063ip0DGMCjKS4Qd9vGVFSFgoU4JKU5soNeqqiwVq66MQc+ByjLxpg4YtenRYIs5bLm97PYhENDob1q3wdvNqiOmrh7dnQN948tqVyIP+jsaR4yaqmCipfKo='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cdf7-298e-7540-9f44-6e92397ec1dc-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'what is langchain-mcp-adapters library'}, 'id': 'call_18189680853129680105', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 199, 'output_tokens'

## Online MCP

In [ ]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [7]:
agent = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=tools,
)

In [8]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response)

{'messages': [HumanMessage(content='What time is it?', additional_kwargs={}, response_metadata={}, id='cc988761-cbbf-408c-9002-d30801bf90b0'),
              AIMessage(content=[], additional_kwargs={'function_call': {'name': 'search_web', 'arguments': '{"query": "current time"}'}, '__gemini_function_call_thought_signatures__': {'call_5274918786210594244': 'EnIKcAFpFH0TdORNPqM0Muhu3lIAqAlc5CTDDOmaXxeaIbZ0Q2KyY0CXfppgMKZXVdpirxqyf/HOX5Rrrs9NpXEtNXcsMpcgD+frnY8insB46jqvYrtMidRidChdc3lvxF99eAH8SiB7eqTlycTyNRzbDcU='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a1cdf8-69d0-7a61-bbb1-e5ddd97710ef-0', tool_calls=[{'name': 'search_web', 'args': {'query': 'current time'}, 'id': 'call_5274918786210594244', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 48, 'output_tokens': 17, 'total_tokens': 65, 'input_token_details': {'cache_read': 0}}),
              Too